# HTTP REST API for GitHub PR Review

This tutorial builds a minimal Flask server with **two APIs** for a code review agent:

1. **Get pull request (files for a given PR number)** – fetch changed files and diffs for a PR  
2. **Add comment** – submit a PR review comment

```
HTTP Client  ──HTTP──►  Flask Server  ──HTTP──►  GitHub API
```

In [ ]:
!pip install flask flask-cors requests python-dotenv

## Step 1: Setup

Load env (optional `GITHUB_TOKEN` for add-comment API) and define helpers for calling the GitHub API.

In [ ]:
import os
import requests
from flask import Flask, jsonify, request
from flask_cors import CORS
from dotenv import load_dotenv

load_dotenv("../../.env.example.json")

app = Flask(__name__)
CORS(app)

GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

print(f"Token loaded: {'Yes' if GITHUB_TOKEN else 'No'}")

def get_headers():
    headers = {"Accept": "application/vnd.github+json"} # media type for GitHub API
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def github_request(method, endpoint, **kwargs):
    return requests.request(method, f"{GITHUB_API_BASE}{endpoint}", headers=get_headers(), **kwargs)

## Step 2: Define the two endpoints

- **GET** ` /repos/<owner>/<repo>/pulls/<pull_number>/files` – get pull request (files/diff for that PR number)  
- **POST** ` /repos/<owner>/<repo>/pulls/<pull_number>/reviews` – add a comment (review) on the PR

In [ ]:
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"})

# 1. Get pull request (files for a given PR number)
@app.route("/repos/<owner>/<repo>/pulls/<int:pull_number>/files", methods=["GET"])
def get_pull_request_files(owner, repo, pull_number):
    """Get files changed in a pull request (diffs and stats)."""
    params = {"per_page": request.args.get("per_page", 50), "page": request.args.get("page")}
    response = github_request("GET", f"/repos/{owner}/{repo}/pulls/{pull_number}/files", params=params)
    return jsonify(response.json()), response.status_code

# 2. Add comment (PR review)
@app.route("/repos/<owner>/<repo>/pulls/<int:pull_number>/reviews", methods=["POST"])
def create_pr_review(owner, repo, pull_number):
    """Create a review on a pull request. Body: {"event": "COMMENT"|"APPROVE"|"REQUEST_CHANGES", "body": "..."}"""
    data = request.get_json()
    if not data or not data.get("event"):
        return jsonify({"error": "event is required", "valid_events": ["APPROVE", "REQUEST_CHANGES", "COMMENT"]}), 400
    payload = {"event": data["event"]}
    if data.get("body"):
        payload["body"] = data["body"]
    response = github_request("POST", f"/repos/{owner}/{repo}/pulls/{pull_number}/reviews", json=payload)
    return jsonify(response.json()), response.status_code

## Step 3: Test the APIs

Test **get pull request files** (no token needed for public repos) and **add comment** (requires `GITHUB_TOKEN` with write access).

In [ ]:
client = app.test_client()

# Health
print("Health:", client.get("/health").get_json())

# 1. Get pull request (files for a given PR number) – use a public repo with an open PR
owner, repo, pull_number = "aishwaryaraimule21", "fifthel-test-repo", 1
resp = client.get(f"/repos/{owner}/{repo}/pulls/{pull_number}/files")
data = resp.get_json()
if isinstance(data, list):
    print(f"PR #{pull_number} files: {len(data)} file(s)")
    for f in data[:3]:
        print(f"  - {f.get('filename')} (+{f.get('additions')}/-{f.get('deletions')})")
else:
    print(f"Get PR files: {resp.status_code}", data if isinstance(data, dict) else "")

# 2. Add comment – only works if GITHUB_TOKEN is set and you have write access to the repo
# Uncomment and set owner/repo/pull_number to a PR you can comment on:
resp = client.post(f"/repos/{owner}/{repo}/pulls/{pull_number}/reviews",
                   json={"event": "COMMENT", "body": "Tutorial check: add comment via API"})
print("Add comment:", resp.status_code, resp.get_json())

## Step 4: Run the server

See `app.py` for the full implementation (including token check for add-comment).

```bash
export GITHUB_TOKEN=your_token   # required for add comment
python app.py
```

**curl examples:**

```bash
# Health
curl http://localhost:5000/health

# Get pull request (files for PR number)
curl http://localhost:5000/repos/octocat/hello-world/pulls/1/files

# Add comment (PR review)
curl -X POST http://localhost:5000/repos/OWNER/REPO/pulls/PR_NUMBER/reviews \
  -H "Content-Type: application/json" \
  -d '{"event": "COMMENT", "body": "Your review text"}'
```